# Notebook 02 — Which words do people really search?

**Project:** Searching for Work in Germany: Google Trends Signals of Labour Migration Interest and Policy Effects (focus: migrant care workers)
**Author:** Tayaphon Rodsai · MSc Data Science for Society and Business, Constructor University
**Supervisors:** Prof. Erkan Gunes, Prof. Jan Lorenz

---

### The question

Notebook 01 selected **9 origin countries**. Now we need the **search keywords** for each country. The keywords should come from **what people really search**, not from our own guesses (supervisor advice).

### Method: two rounds of related queries ("snowball")

1. **Round 1 (this notebook, done):** for each country, Google Trends lists the **top** and **rising related queries** for the topic *Germany*. We keep only the queries about **work, care, training, language and visa**.
2. **Round 2 (next):** the best keywords from round 1 (plus a few seed words in the local language) are searched again, and **their** related queries give more specific keywords.
3. **Check:** social media and native speakers confirm that the words are really used.

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
GT_CHECK = RAW / "gt_check"

CODES = {
    "IN": "India", "TR": "Türkiye", "VN": "Viet Nam", "MA": "Morocco", "TN": "Tunisia",
    "PH": "Philippines", "ID": "Indonesia", "BA": "Bosnia and Herzegovina", "CM": "Cameroon",
}

## 2. Load all round-1 related queries

Source: Google Trends, topic *Germany – Country in Europe*, Jan 2015 – Oct 2026, downloaded 13 Nov 2026 (see Notebook 01). Each country has a **top** list (most searched) and a **rising** list (fastest growing), 50 queries each.

In [ ]:
frames = []
for code_, country in CODES.items():
    for list_type in ["top", "rising"]:
        q = pd.read_csv(GT_CHECK / f"{code_}_topic_germany_{list_type}.csv")
        q["code"], q["country"], q["list"] = code_, country, list_type
        frames.append(q)

queries = pd.concat(frames, ignore_index=True)
queries["query"] = queries["query"].astype(str).str.lower().str.strip()
print("Queries loaded:", len(queries), "| unique:", queries["query"].nunique())
queries.head()

## 3. Sort queries into themes

Most queries are **noise** for this project (football, time zones, shopping, COVID, even casino and lottery spam). We look for **word stems** that belong to five themes. The stems are written in the languages that appear in the data (English, French, Arabic, Turkish, Vietnamese, Indonesian, Bosnian, German).

| Theme | Meaning | Examples of stems |
|---|---|---|
| **care** | nurse, caregiver | nurse, infirmier, hemşire, điều dưỡng, perawat, medicinska sestra, تمريض |
| **training** | vocational training | ausbildung, du học nghề, التكوين المهني |
| **work** | jobs, salaries, recruitment | job, posao, iş ilan, işçi alımı, emploi, việc làm, kerja, indeed, salaire |
| **language** | learning German | german language, tiếng đức, njemački jezik, almanca, goethe |
| **visa** | migration process | visa, vize, viza, tls, vfs, idata, ambassade, embassy, make it in germany |

A query gets the **first** theme that matches, in the order of the table (so *salaire infirmier* = **care**, not work). **University study** is recorded separately, because it is not a care route (see Notebook 01).

> The automatic sorting is only a **first filter**. Every match is checked by hand (Section 4).

In [ ]:
THEMES = {
    "care": ["nurse", "nursing", "caregiver", "infirmier", "infirmière", "aide-soignant", "hemşire", "hasta bakıcı",
             "điều dưỡng", "perawat", "medicinska sestra", "medicinske sestre", "njegovatelj", "تمريض", "ممرض", "pflege"],
    "training": ["ausbildung", "du học nghề", "التكوين المهني", "formation professionnelle", "mesleki eğitim"],
    "work": ["job", "work in", "working in", "posao", "poslovi", "iş ilan", "işçi alımı", "iş başvuru", "asgari ücret",
             "emploi", "travail", "travailler", "việc làm", "xuất khẩu lao động", "kerja", "indeed", "salaire", "salary", "العمل"],
    "language": ["german language", "learn german", "tiếng đức", "njemački jezik", "njemacki jezik", "učenje njemačkog",
                 "almanca", "bahasa jerman", "goethe", "apprendre l'allemand", "اللغة الالمانية", "اللغة الألمانية"],
    "visa": ["visa", "vize", "viza", "vizu", "tls", "vfs", "idata", "ambassade", "ambasada", "embassy", "konsolos",
             "make it in germany", "einreise", "plava karta", "blue card", "chancenkarte", "opportunity card",
             "fırsat kartı", "spajanje porodice"],
    "study (not used)": ["study in", "etudier", "étudier", "du học", "الدراسة", "universit", "ms in germany"],
}


def find_theme(query):
    for theme, stems in THEMES.items():
        if any(stem in query for stem in stems):
            return theme
    return None


queries["theme"] = queries["query"].map(find_theme)
relevant = queries.dropna(subset=["theme"])
print(f"Relevant: {len(relevant)} of {len(queries)} queries ({len(relevant) / len(queries):.0%})")

### How many relevant queries per country and theme?

In [ ]:
overview = pd.crosstab(relevant["country"], relevant["theme"]).reindex(CODES.values()).fillna(0).astype(int)
overview["total"] = overview.sum(axis=1)
overview

## 4. The candidate keywords (check by hand)

One row per country and query (a query can appear in both the top and the rising list). Read every row and decide if it really fits.

In [ ]:
candidates = (relevant
              .groupby(["country", "theme", "query"], as_index=False)
              .agg(lists=("list", lambda s: " + ".join(sorted(set(s))))))
candidates = candidates.sort_values(["country", "theme", "query"])

pd.set_option("display.max_rows", 200)
candidates

> **Finding 1 — useful keywords exist, but they are rare.** Only a small share of the related queries are about work, care, training, language or visa. The rest is mainly football, time zones and shopping.
>
> **Finding 2 — the countries differ a lot.**
> - **Türkiye** and **Bosnia and Herzegovina** have many **work** queries (*almanya iş ilanları*, *posao u njemačkoj*).
> - **Tunisia** has the clearest **care** query (*salaire infirmier en allemagne*) and a **job-seeker visa** query (*visa recherche d'emploi allemagne*), which is close to the Chancenkarte.
> - **Morocco, Tunisia and the Philippines** use the German word **"ausbildung"** directly.
> - **Indonesia** has almost nothing useful (football and lottery), so round 2 is especially important there.
>
> **Finding 3 — some keywords work across countries.** *Make it in Germany* (the official portal), *ausbildung*, and the visa centres (**TLS, VFS, iData**) appear in several countries. Searches for visa appointments are a strong sign of a **real plan** to move.

## 5. Save round 1

In [ ]:
candidates.to_csv(PROCESSED / "keywords_round1_candidates.csv", index=False)
print("Saved:", PROCESSED / "keywords_round1_candidates.csv")

---
## 6. Next: round 2 (seed keywords)

Round 1 found **no care keywords** in most countries, because the topic *Germany* is too broad. In round 2 we search **specific seed keywords** and download **their** related queries.

Seeds come from two places:
1. the best round-1 keywords (marked ✔︎), and
2. common local words for *nurse in Germany*, *Ausbildung in Germany* and *work in Germany* (marked ✎). These are **only starting points**: a seed is kept only if Google Trends has data for it and its related queries confirm it.

| Code | Care seed | Training seed | Work seed |
|---|---|---|---|
| VN | điều dưỡng đức ✎ | du học nghề đức ✎ | xuất khẩu lao động đức ✎ |
| PH | nurse germany ✎ | ausbildung in germany ✔︎ | jobs in germany ✎ |
| IN | nursing jobs in germany ✎ | ausbildung in germany ✎ | jobs in germany ✔︎ |
| TR | almanya hemşire ✎ | almanya ausbildung ✎ | almanya iş ilanları ✔︎ |
| MA | infirmier allemagne ✎ | ausbildung ✔︎ | travail en allemagne ✎ |
| TN | salaire infirmier en allemagne ✔︎ | ausbildung en allemagne ✔︎ | travail en allemagne ✎ |
| ID | perawat jerman ✎ | ausbildung jerman ✎ | kerja di jerman ✎ |
| BA | medicinska sestra njemačka ✎ | ausbildung njemačka ✎ | posao u njemačkoj ✔︎ |
| CM | infirmier allemagne ✎ | ausbildung allemagne ✎ | travailler en allemagne ✎ |

**Plan:** start with the **3 pilot countries (VN, PH, IN)**. For each seed: Google Trends → *search term* → country → 1 Jan 2015 – 31 Oct 2026 → download **top** and **rising** related queries → save in `data/raw/gt_keywords/round2/` as `VN_care_top.csv`, `VN_care_rising.csv`, and so on.